# 🎗️ Cancer Patient ML Prediction

**Three regression targets predicted from the same dataset:**

| # | Target | Models |
|---|---|---|
| 1 | `Treatment_Cost_USD` | Linear, Decision Tree, Random Forest, KNN, SVR, Gradient Boosting |
| 2 | `Target_Severity_Score` | Linear, KNN, SVR, Decision Tree, Random Forest, Gradient Boosting |
| 3 | `Survival_Years` | Linear, Decision Tree, Random Forest, KNN, SVR, Gradient Boosting |

**Dataset:** Global Cancer Patients  
**Author:** Rudra Dahikar — https://www.kaggle.com/rudradahikar

## 1. Setup & Imports

In [ ]:
import pandas as pd
import numpy as np
import joblib
import os
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR

import matplotlib.pyplot as plt
matplotlib.rcParams = matplotlib.rcParamsDefault
import matplotlib

os.makedirs('/kaggle/working/models', exist_ok=True)
print('All libraries loaded!')

## 2. Load & Explore Dataset

In [ ]:
DATA_PATH = '/kaggle/input/cancer-prediction-dataset/cancer.csv'
df_raw = pd.read_csv(DATA_PATH)
print('Shape:', df_raw.shape)
df_raw.head()

In [ ]:
df_raw.info()

In [ ]:
df_raw.describe()

In [ ]:
print('Missing values:\n', df_raw.isnull().sum())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].hist(df_raw['Treatment_Cost_USD'].dropna(), bins=40, color='steelblue', edgecolor='white')
axes[0].set_title('Treatment Cost (USD)')
axes[1].hist(df_raw['Target_Severity_Score'].dropna(), bins=40, color='tomato', edgecolor='white')
axes[1].set_title('Severity Score')
axes[2].hist(df_raw['Survival_Years'].dropna(), bins=40, color='seagreen', edgecolor='white')
axes[2].set_title('Survival Years')
plt.suptitle('Target Variable Distributions', fontsize=13)
plt.tight_layout()
plt.savefig('/kaggle/working/target_distributions.png', dpi=150)
plt.show()

## 3. Helper Functions

In [ ]:
all_results = {}

def evaluate(target, name, model, X_tr, y_tr, X_te, y_te, save_dir):
    model.fit(X_tr, y_tr)
    y_pred = model.predict(X_te)
    r2  = r2_score(y_te, y_pred)
    mse = mean_squared_error(y_te, y_pred)
    mae = mean_absolute_error(y_te, y_pred)
    if target not in all_results:
        all_results[target] = {}
    all_results[target][name] = {
        'R2 Score (%)': round(r2 * 100, 3),
        'MSE': round(mse, 3),
        'MAE': round(mae, 3)
    }
    os.makedirs(save_dir, exist_ok=True)
    joblib.dump(model, f'{save_dir}/model.dat')
    print(f'  [{name:20s}]  R2={r2*100:.2f}%  MSE={mse:.2f}  MAE={mae:.2f}')
    return y_pred

def results_table(target):
    return pd.DataFrame(all_results[target]).T.sort_values('R2 Score (%)', ascending=False)

def comparison_chart(target, color):
    df_r = results_table(target)
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    df_r['R2 Score (%)'].plot(kind='barh', ax=axes[0], color=color, edgecolor='white')
    axes[0].set_title('R2 Score (%)')
    df_r['MSE'].plot(kind='barh', ax=axes[1], color=color, edgecolor='white')
    axes[1].set_title('MSE')
    df_r['MAE'].plot(kind='barh', ax=axes[2], color=color, edgecolor='white')
    axes[2].set_title('MAE')
    plt.suptitle(f'Model Comparison — {target}', fontsize=13)
    plt.tight_layout()
    safe = target.replace(' ', '_')
    plt.savefig(f'/kaggle/working/{safe}_comparison.png', dpi=150)
    plt.show()
    return df_r

---
# Part A — Treatment Cost Prediction
**Target:** `Treatment_Cost_USD`

In [ ]:
df_cost = df_raw.copy().dropna()
df_cost = df_cost.drop(['Patient_ID', 'Survival_Years'], axis=1)

enc_cost = LabelEncoder()
for col in ['Gender', 'Country_Region', 'Cancer_Type', 'Cancer_Stage']:
    df_cost[col] = enc_cost.fit_transform(df_cost[col])

X_cost = df_cost.drop('Treatment_Cost_USD', axis=1)
y_cost = df_cost['Treatment_Cost_USD']

X_cost_tr, X_cost_te, y_cost_tr, y_cost_te = train_test_split(
    X_cost, y_cost, test_size=0.2, random_state=10)

print(f'Train: {X_cost_tr.shape}  Test: {X_cost_te.shape}')

In [ ]:
print('=== Treatment Cost USD ===')
BASE = '/kaggle/working/models/cost'

evaluate('Treatment_Cost_USD', 'LinearRegression',
         LinearRegression(), X_cost_tr, y_cost_tr, X_cost_te, y_cost_te, f'{BASE}/linear')
evaluate('Treatment_Cost_USD', 'DecisionTree',
         DecisionTreeRegressor(max_depth=15, random_state=42),
         X_cost_tr, y_cost_tr, X_cost_te, y_cost_te, f'{BASE}/decisiontree')
evaluate('Treatment_Cost_USD', 'RandomForest',
         RandomForestRegressor(n_estimators=100, max_depth=15, random_state=42),
         X_cost_tr, y_cost_tr, X_cost_te, y_cost_te, f'{BASE}/randomforest')
evaluate('Treatment_Cost_USD', 'KNN',
         KNeighborsRegressor(), X_cost_tr, y_cost_tr, X_cost_te, y_cost_te, f'{BASE}/knn')
evaluate('Treatment_Cost_USD', 'SVR',
         SVR(kernel='rbf'), X_cost_tr, y_cost_tr, X_cost_te, y_cost_te, f'{BASE}/svr')
evaluate('Treatment_Cost_USD', 'GradientBoosting',
         GradientBoostingRegressor(max_depth=5, n_estimators=100, random_state=42),
         X_cost_tr, y_cost_tr, X_cost_te, y_cost_te, f'{BASE}/gradientboosting')

In [ ]:
cost_results = comparison_chart('Treatment_Cost_USD', 'steelblue')
cost_results

In [ ]:
best_cost = cost_results.index[0]
best_cost_model = joblib.load(f'/kaggle/working/models/cost/{best_cost.lower()}/model.dat')
pred_cost = best_cost_model.predict(X_cost_te)

plt.figure(figsize=(7, 5))
plt.scatter(y_cost_te, pred_cost, alpha=0.35, s=12, color='steelblue')
mn, mx = y_cost_te.min(), y_cost_te.max()
plt.plot([mn, mx], [mn, mx], 'r--', lw=2, label='Perfect fit')
plt.xlabel('Actual Cost (USD)')
plt.ylabel('Predicted Cost (USD)')
plt.title(f'Actual vs Predicted — {best_cost}')
plt.legend()
plt.tight_layout()
plt.savefig('/kaggle/working/cost_actual_vs_predicted.png', dpi=150)
plt.show()

---
# Part B — Cancer Severity Score Prediction
**Target:** `Target_Severity_Score`

In [ ]:
df_sev = df_raw.copy().dropna()
df_sev = df_sev.drop(['Patient_ID', 'Treatment_Cost_USD', 'Survival_Years'], axis=1)

enc_sev = {}
os.makedirs('/kaggle/working/models/severity', exist_ok=True)
for col in ['Gender', 'Country_Region', 'Cancer_Type', 'Cancer_Stage']:
    le = LabelEncoder()
    df_sev[col] = le.fit_transform(df_sev[col])
    enc_sev[col] = le
    joblib.dump(le, f'/kaggle/working/models/severity/{col}_encoder.enc')

X_sev = df_sev.drop('Target_Severity_Score', axis=1)
y_sev = df_sev['Target_Severity_Score']

X_sev_tr, X_sev_te, y_sev_tr, y_sev_te = train_test_split(
    X_sev, y_sev, test_size=0.2, random_state=10)

print(f'Train: {X_sev_tr.shape}  Test: {X_sev_te.shape}')

In [ ]:
print('=== Target Severity Score ===')
BASE_SEV = '/kaggle/working/models/severity'

evaluate('Target_Severity_Score', 'LinearRegression',
         LinearRegression(), X_sev_tr, y_sev_tr, X_sev_te, y_sev_te, f'{BASE_SEV}/linear')
evaluate('Target_Severity_Score', 'KNN',
         KNeighborsRegressor(), X_sev_tr, y_sev_tr, X_sev_te, y_sev_te, f'{BASE_SEV}/knn')
evaluate('Target_Severity_Score', 'SVR',
         SVR(kernel='rbf'), X_sev_tr, y_sev_tr, X_sev_te, y_sev_te, f'{BASE_SEV}/svr')
evaluate('Target_Severity_Score', 'DecisionTree',
         DecisionTreeRegressor(max_depth=15, random_state=42),
         X_sev_tr, y_sev_tr, X_sev_te, y_sev_te, f'{BASE_SEV}/decisiontree')
evaluate('Target_Severity_Score', 'RandomForest',
         RandomForestRegressor(n_estimators=100, max_depth=15, random_state=42),
         X_sev_tr, y_sev_tr, X_sev_te, y_sev_te, f'{BASE_SEV}/randomforest')
evaluate('Target_Severity_Score', 'GradientBoosting',
         GradientBoostingRegressor(max_depth=5, n_estimators=100, random_state=42),
         X_sev_tr, y_sev_tr, X_sev_te, y_sev_te, f'{BASE_SEV}/gradientboosting')

In [ ]:
sev_results = comparison_chart('Target_Severity_Score', 'tomato')
sev_results

In [ ]:
best_sev = sev_results.index[0]
best_sev_model = joblib.load(f'/kaggle/working/models/severity/{best_sev.lower()}/model.dat')
pred_sev = best_sev_model.predict(X_sev_te)

plt.figure(figsize=(7, 5))
plt.scatter(y_sev_te, pred_sev, alpha=0.35, s=12, color='tomato')
mn, mx = y_sev_te.min(), y_sev_te.max()
plt.plot([mn, mx], [mn, mx], 'b--', lw=2, label='Perfect fit')
plt.xlabel('Actual Severity Score')
plt.ylabel('Predicted Severity Score')
plt.title(f'Actual vs Predicted — {best_sev}')
plt.legend()
plt.tight_layout()
plt.savefig('/kaggle/working/severity_actual_vs_predicted.png', dpi=150)
plt.show()

### B4. Live Prediction Pipeline (Severity → Cost)

In [ ]:
sample = {
    'Age': 45, 'Gender': 'Male', 'Country_Region': 'India', 'Year': 2022,
    'Genetic_Risk': 5.0, 'Air_Pollution': 3.5, 'Alcohol_Use': 2.0,
    'Smoking': 1.5, 'Obesity_Level': 4.0, 'Cancer_Type': 'Lung', 'Cancer_Stage': 'Stage II'
}

sev_row = [
    sample['Age'],
    enc_sev['Gender'].transform([sample['Gender']])[0],
    enc_sev['Country_Region'].transform([sample['Country_Region']])[0],
    sample['Year'], sample['Genetic_Risk'], sample['Air_Pollution'],
    sample['Alcohol_Use'], sample['Smoking'], sample['Obesity_Level'],
    enc_sev['Cancer_Type'].transform([sample['Cancer_Type']])[0],
    enc_sev['Cancer_Stage'].transform([sample['Cancer_Stage']])[0]
]

sev_score = best_sev_model.predict([sev_row])[0]
print(f'Predicted Severity Score : {sev_score:.4f}')

cost_row = sev_row + [sev_score]
cost_pred = best_cost_model.predict([cost_row])[0]
print(f'Predicted Treatment Cost : ${cost_pred:,.2f} USD')

---
# Part C — Survival Years Prediction
**Target:** `Survival_Years`

In [ ]:
drop_cols_years = ['Patient_ID', 'Age', 'Gender', 'Country_Region', 'Year',
                   'Genetic_Risk', 'Air_Pollution', 'Alcohol_Use', 'Smoking',
                   'Obesity_Level', 'Treatment_Cost_USD']

df_yrs = df_raw.copy().dropna()
df_yrs = df_yrs.drop(drop_cols_years, axis=1)

enc_yr = LabelEncoder()
df_yrs['Cancer_Type']  = enc_yr.fit_transform(df_yrs['Cancer_Type'])
df_yrs['Cancer_Stage'] = enc_yr.fit_transform(df_yrs['Cancer_Stage'])

X_yrs = df_yrs.drop('Survival_Years', axis=1)
y_yrs = df_yrs['Survival_Years']

X_yrs_tr, X_yrs_te, y_yrs_tr, y_yrs_te = train_test_split(
    X_yrs, y_yrs, test_size=0.2, random_state=10)

print(f'Train: {X_yrs_tr.shape}  Test: {X_yrs_te.shape}')
X_yrs.head()

In [ ]:
print('=== Survival Years ===')
BASE_YRS = '/kaggle/working/models/years'

evaluate('Survival_Years', 'LinearRegression',
         LinearRegression(), X_yrs_tr, y_yrs_tr, X_yrs_te, y_yrs_te, f'{BASE_YRS}/linear')
evaluate('Survival_Years', 'DecisionTree',
         DecisionTreeRegressor(random_state=42),
         X_yrs_tr, y_yrs_tr, X_yrs_te, y_yrs_te, f'{BASE_YRS}/decisiontree')
evaluate('Survival_Years', 'RandomForest',
         RandomForestRegressor(n_estimators=100, random_state=42),
         X_yrs_tr, y_yrs_tr, X_yrs_te, y_yrs_te, f'{BASE_YRS}/randomforest')
evaluate('Survival_Years', 'KNN',
         KNeighborsRegressor(), X_yrs_tr, y_yrs_tr, X_yrs_te, y_yrs_te, f'{BASE_YRS}/knn')
evaluate('Survival_Years', 'SVR',
         SVR(kernel='rbf'), X_yrs_tr, y_yrs_tr, X_yrs_te, y_yrs_te, f'{BASE_YRS}/svr')
evaluate('Survival_Years', 'GradientBoosting',
         GradientBoostingRegressor(n_estimators=100, random_state=42),
         X_yrs_tr, y_yrs_tr, X_yrs_te, y_yrs_te, f'{BASE_YRS}/gradientboosting')

In [ ]:
yrs_results = comparison_chart('Survival_Years', 'seagreen')
yrs_results

In [ ]:
best_yrs = yrs_results.index[0]
best_yrs_model = joblib.load(f'/kaggle/working/models/years/{best_yrs.lower()}/model.dat')
pred_yrs = best_yrs_model.predict(X_yrs_te)

plt.figure(figsize=(7, 5))
plt.scatter(y_yrs_te, pred_yrs, alpha=0.35, s=12, color='seagreen')
mn, mx = y_yrs_te.min(), y_yrs_te.max()
plt.plot([mn, mx], [mn, mx], 'r--', lw=2, label='Perfect fit')
plt.xlabel('Actual Survival Years')
plt.ylabel('Predicted Survival Years')
plt.title(f'Actual vs Predicted — {best_yrs}')
plt.legend()
plt.tight_layout()
plt.savefig('/kaggle/working/years_actual_vs_predicted.png', dpi=150)
plt.show()

---
# Part D — Overall Summary

In [ ]:
summary_rows = []
for target, models in all_results.items():
    for model, metrics in models.items():
        summary_rows.append({'Target': target, 'Model': model, **metrics})

summary_df = pd.DataFrame(summary_rows)
summary_df = summary_df.sort_values(['Target', 'R2 Score (%)'], ascending=[True, False])
summary_df.to_csv('/kaggle/working/all_results_summary.csv', index=False)
print('Saved: /kaggle/working/all_results_summary.csv')
summary_df

In [ ]:
print('\n=== BEST MODEL PER TARGET ===')
for target in all_results:
    best = max(all_results[target], key=lambda m: all_results[target][m]['R2 Score (%)'])
    r2 = all_results[target][best]['R2 Score (%)']
    print(f'  {target:35s} -> {best:20s}  R2={r2:.2f}%')

In [ ]:
rf_cost = joblib.load('/kaggle/working/models/cost/randomforest/model.dat')
imp = pd.Series(rf_cost.feature_importances_, index=X_cost.columns).sort_values()

plt.figure(figsize=(8, 5))
imp.plot(kind='barh', color='steelblue', edgecolor='white')
plt.title('Feature Importances — Random Forest (Treatment Cost)')
plt.xlabel('Importance')
plt.tight_layout()
plt.savefig('/kaggle/working/feature_importance_cost.png', dpi=150)
plt.show()

## Summary

Three regression pipelines on the Global Cancer Patients dataset:

| Target | Features Used | Models Tested |
|---|---|---|
| Treatment_Cost_USD | All 12 features | 6 |
| Target_Severity_Score | 11 features (no cost/survival) | 6 |
| Survival_Years | Cancer_Type, Cancer_Stage, Severity | 6 |

All trained models saved to `/kaggle/working/models/` — downloadable from **Output** tab.